In [3]:
from sklearn.preprocessing import OrdinalEncoder
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

import time
import numpy as np

# Preprocessing

In [4]:
# Encode a numeric column as zscores
def zscore_normalization(df, name, mean=None, sd=None):
    if mean is None:
        mean = df[name].mean()

    if sd is None:
        sd = df[name].std()

    df[name] = (df[name] - mean) / sd
def encode_text(df, name):
    enc = OrdinalEncoder()
    # dummies = pd.get_dummies(df[name])
    data = enc.fit_transform(df[name].values.reshape(-1,1))
    df[name]=data.flatten()
def preprocess(df):
    #Encoding and Normalizing Features
    zscore_normalization(df, 'duration')
    encode_text(df, 'protocol_type')
    encode_text(df, 'service')
    encode_text(df, 'flag')
    zscore_normalization(df, 'src_bytes')
    zscore_normalization(df, 'dst_bytes')
    encode_text(df, 'land')
    zscore_normalization(df, 'wrong_fragment')
    zscore_normalization(df, 'urgent')
    zscore_normalization(df, 'hot')
    zscore_normalization(df, 'num_failed_logins')
    encode_text(df, 'logged_in')
    zscore_normalization(df, 'num_compromised')
    zscore_normalization(df, 'root_shell')
    zscore_normalization(df, 'su_attempted')
    zscore_normalization(df, 'num_root')
    zscore_normalization(df, 'num_file_creations')
    zscore_normalization(df, 'num_shells')
    zscore_normalization(df, 'num_access_files')
    zscore_normalization(df, 'num_outbound_cmds')
    encode_text(df, 'is_host_login')
    encode_text(df, 'is_guest_login')
    zscore_normalization(df, 'count')
    zscore_normalization(df, 'srv_count')
    zscore_normalization(df, 'serror_rate')
    zscore_normalization(df, 'srv_serror_rate')
    zscore_normalization(df, 'rerror_rate')
    zscore_normalization(df, 'srv_rerror_rate')
    zscore_normalization(df, 'same_srv_rate')
    zscore_normalization(df, 'diff_srv_rate')
    zscore_normalization(df, 'srv_diff_host_rate')
    zscore_normalization(df, 'dst_host_count')
    zscore_normalization(df, 'dst_host_srv_count')
    zscore_normalization(df, 'dst_host_same_srv_rate')
    zscore_normalization(df, 'dst_host_diff_srv_rate')
    zscore_normalization(df, 'dst_host_same_src_port_rate')
    zscore_normalization(df, 'dst_host_srv_diff_host_rate')
    zscore_normalization(df, 'dst_host_serror_rate')
    zscore_normalization(df, 'dst_host_srv_serror_rate')
    zscore_normalization(df, 'dst_host_rerror_rate')
    zscore_normalization(df, 'dst_host_srv_rerror_rate')
    df.dropna(inplace=True,axis=1)
    for col in df.columns:
        if len(df[col].unique()) == 1:
            df.drop(col, inplace=True,axis=1)
    # Encoding outcomes to binary
    df.loc[df['outcome'] != "normal.", 'outcome']  = 1
    df.loc[df['outcome'] == "normal.", 'outcome']  = 0
    # Feature selection
    for num in correlation:
     if num >= -0.05 and num <= 0.05:
         df.drop(df.columns[row], axis=1, inplace=True)
         row += 1

# Model Training and Testing

In [10]:
def training_basic_classifier(df, X_train, X_test, y_train, y_test): 
    # Logistic regression
    start_train = time.time()
    lrc = LogisticRegression(random_state=0, max_iter=1000)
    
    parameters = {
        'C':[0.01, 0.1, 1, 10],
        'solver':['lbfgs', 'newton-cholesky']
    }
    
    grid_search = GridSearchCV(estimator=lrc, param_grid=parameters, cv=5, scoring='accuracy', verbose=1).fit(X_train, y_train)
    
    best_lr=grid_search.best_estimator_
    end_train = time.time()
    
    start_test = time.time()
    ypredlr = best_lr.predict(X_test)
    end_test = time.time()
    
    lr_train_time = end_train-start_train
    lr_test_time = end_test-start_test

    accuracy = accuracy_score(y_test, ypredlr)
    
    f1 = f1_score(y_test, ypredlr)
    print(f"LR Accuracy: {accuracy}")
    print(f"LR F1 Score: {f1}")
    # Random Forest Classifier
    start_train = time.time()
    rfc = RandomForestClassifier()
    rfc.fit(X_train, y_train)
    end_train = time.time()
    
    start_test = time.time()
    y_pred2=rfc.predict(X_test)
    end_test = time.time()
    
    RFC_train_time, end_train-start_train
    RFC_test_time = end_test-start_test
    accuracy = accuracy_score(y_test, y_pred2)
    f1 = f1_score(y_test, y_pred2)
    print(f"RFC Accuracy: {accuracy}")
    print(f"RFC F1 Score: {f1}")
    # Decision Trees
    start_train = time.time()
    dtc = DecisionTreeClassifier()
    dtc.fit(X_train, y_train)
    end_train = time.time()

    start_test = time.time()
    y_pred3=dtc.predict(X_test)
    end_test = time.time()

    dt_train = end_train-start_train
    dt_test = end_test-start_test
    accuracy = accuracy_score(y_test,y_pred3)
    f1 = f1_score(y_test, y_pred3)
    print(f"DTC Accuracy: {accuracy}")
    print(f"DTC F1 Score: {f1}")
    # SVM
    start_train = time.time()
    svc = SVC()
    svc.fit(X_train, y_train)
    end_train = time.time()
    
    start_test = time.time()
    y_pred4=svc.predict(X_test)
    end_test = time.time()
    
    svm_train = end_train-start_train
    svm_test = end_test-start_test
    accuracy = accuracy_score(y_test,y_pred4)
    f1 = f1_score(y_test, y_pred4)
    print(f"SVM Accuracy: {accuracy}")
    print(f"SVM F1 Score: {f1}")
    #Gradient Boost
    start_train = time.time()
    gbc = GradientBoostingClassifier()
    gbc.fit(X_train, y_train)
    end_train = time.time()
    
    start_test = time.time()
    y_pred5=gbc.predict(X_test)
    end_test = time.time()
    
    gradient_train = end_train-start_train
    gradient_test = end_test-start_test
    accuracy = accuracy_score(y_test,y_pred5)
    f1 = f1_score(y_test, y_pred5)
    print(f"Gradient Accuracy: {accuracy}")
    print(f"Gradient F1 Score: {f1}")

    #Gaussian Naive Bayes
    start_train = time.time()
    gnb = GaussianNB()
    gnb.fit(X_train, y_train)
    end_train = time.time()
    
    start_test = time.time()
    y_pred6=gnb.predict(X_test)
    end_test = time.time()
    
    gnb_train = end_train-start_train
    gnb_test = end_test-start_test
    accuracy = accuracy_score(y_test,y_pred6)
    f1 = f1_score(y_test, y_pred6)
    print(f"SVM Accuracy: {accuracy}")
    print(f"SVM F1 Score: {f1}")